# Silver Layer: Flight Booking Table Cleaning & Validation

Cleanses `skyledger_airlines.bronze.flight_booking` into `skyledger_airlines.silver.flight_booking`.
Applies primary key validation (§3.1), numeric range validation (§3.2), date range validation (§3.3), and referential integrity checks (§3.4).
Invalid records are quarantined in `silver.flight_booking_quarantine` with comma-separated rejection reasons.
Includes Bronze-Silver-Quarantine reconciliation and a data-quality report.

**Relationships:** `passenger` -> `booking` -> `flight_booking` (core booking fact chain).
`route` -> `flight_booking` (product sold on each segment).
`airline` -> `flight_booking` (carrier operating each segment).

**Composite PK (§3.1):** `fb_bookingid` + `fb_linenumber` -- both must be not null and the pair must be unique.

**Numeric rules (§3.2):** `fb_ticketprice` > 0, `fb_discount` in [0, 1], `fb_seats` >= 1.

**Date rules (§3.3):** All dates within 1994-01-01 to 1998-12-31 inclusive.
Cross-column: `fb_departuredate` >= `bk_bookingdate` (from parent booking), `fb_arrivaldate` >= `fb_departuredate`.

**FK rules (§3.4):** `fb_bookingid` -> `silver.booking`, `fb_routeid` -> `silver.route`, `fb_airlineid` -> `silver.airline`.
Parent-before-child order: booking, route, and airline are validated before flight_booking.

In [0]:
%sql
SELECT * FROM skyledger_airlines.bronze.flight_booking LIMIT 10

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql.functions import col, when, lit, count as _count, concat_ws, to_date
from pyspark.sql.window import Window

bronze_fb = spark.table("skyledger_airlines.bronze.flight_booking")

# Drop metadata columns: _ingestion_timestamp, _source_file, _pipeline_run_id
# Cast IDs to INT, measures to DOUBLE, dates to DATE, strings remain strings
fb_typed = bronze_fb.select(
    col("fb_bookingid").cast("int").alias("fb_bookingid"),
    col("fb_routeid").cast("int").alias("fb_routeid"),
    col("fb_airlineid").cast("int").alias("fb_airlineid"),
    col("fb_linenumber").cast("int").alias("fb_linenumber"),
    col("fb_seats").cast("int").alias("fb_seats"),
    col("fb_ticketprice").cast("double").alias("fb_ticketprice"),
    col("fb_discount").cast("double").alias("fb_discount"),
    col("fb_tax").cast("double").alias("fb_tax"),
    col("fb_refundflag").cast("string").alias("fb_refundflag"),
    col("fb_segmentstatus").cast("string").alias("fb_segmentstatus"),
    to_date(col("fb_departuredate")).alias("fb_departuredate"),
    to_date(col("fb_commitdate")).alias("fb_commitdate"),
    to_date(col("fb_arrivaldate")).alias("fb_arrivaldate"),
    col("fb_cabinclass").cast("string").alias("fb_cabinclass"),
    col("fb_comment").cast("string").alias("fb_comment"),
)

bronze_count = fb_typed.count()
print(f"Bronze flight_booking rows read: {bronze_count}")
display(fb_typed.limit(10))

In [0]:
# ── Step 2: Primary Key Validation (§3.1) ──
# Composite PK: fb_bookingid + fb_linenumber — both must be NOT NULL and the pair must be UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK

from pyspark.sql.functions import row_number

fb_flagged = fb_typed.withColumn(
    "is_null_pk",
    when(col("fb_bookingid").isNull() | col("fb_linenumber").isNull(), lit(True)).otherwise(lit(False)),
)

# Window for duplicate detection and selection
pk_window = Window.partitionBy("fb_bookingid", "fb_linenumber").orderBy(lit(1))

# Assign row_number to keep only one per composite PK
fb_flagged = fb_flagged.withColumn(
    "_pk_count", _count("*").over(Window.partitionBy("fb_bookingid", "fb_linenumber"))
).withColumn(
    "is_duplicate_pk",
    when(
        col("fb_bookingid").isNotNull() & col("fb_linenumber").isNotNull() & (col("_pk_count") > 1),
        lit(True),
    ).otherwise(lit(False)),
).withColumn(
    "_pk_row_number", row_number().over(pk_window)
)

# Mark as duplicate except first valid row (null PK and duplicate PK get flagged as before)
fb_flagged = fb_flagged.withColumn(
    "is_duplicate_pk",
    when((col("is_duplicate_pk") == True) & (col("_pk_row_number") > 1), lit(True)).otherwise(col("is_duplicate_pk")),
)

null_pk_count = fb_flagged.filter(col("is_null_pk")).count()
duplicate_pk_count = fb_flagged.filter((col("is_duplicate_pk") == True)).count()

print(f"NULL_PK failures:      {null_pk_count}")
print(f"DUPLICATE_PK failures: {duplicate_pk_count}")

In [0]:
# ── Step 3: Numeric Range Validation (§3.2) ──
# Rule: fb_ticketprice must be strictly greater than zero.
# Rule: fb_discount must lie in the closed interval [0, 1].
# Rule: fb_seats must be a positive integer (1 or more).
# Failure codes: INVALID_TICKETPRICE, INVALID_DISCOUNT, INVALID_SEATS

fb_flagged = fb_flagged.withColumn(
    "is_invalid_ticketprice",
    when(col("fb_ticketprice").isNull() | (col("fb_ticketprice") <= 0), lit(True)).otherwise(lit(False)),
)

fb_flagged = fb_flagged.withColumn(
    "is_invalid_discount",
    when(col("fb_discount").isNull() | (col("fb_discount") < 0) | (col("fb_discount") > 1), lit(True)).otherwise(lit(False)),
)

fb_flagged = fb_flagged.withColumn(
    "is_invalid_seats",
    when(col("fb_seats").isNull() | (col("fb_seats") < 1), lit(True)).otherwise(lit(False)),
)

invalid_ticketprice_count = fb_flagged.filter(col("is_invalid_ticketprice")).count()
invalid_discount_count = fb_flagged.filter(col("is_invalid_discount")).count()
invalid_seats_count = fb_flagged.filter(col("is_invalid_seats")).count()

print(f"INVALID_TICKETPRICE failures: {invalid_ticketprice_count}")
print(f"INVALID_DISCOUNT failures:   {invalid_discount_count}")
print(f"INVALID_SEATS failures:       {invalid_seats_count}")

In [0]:
# ── Step 4: Date Range Validation (§3.3) ──
# Global operational window: all dates must fall between 1994-01-01 and 1998-12-31 inclusive.
# Cross-column: fb_arrivaldate must be on or after fb_departuredate (physically impossible otherwise).
# Failure codes: INVALID_DATE_DEPARTURE, INVALID_DATE_COMMIT, INVALID_DATE_ARRIVAL, ARRIVAL_BEFORE_DEPARTURE

from pyspark.sql.functions import lit as _lit
min_date = _lit("1994-01-01").cast("date")
max_date = _lit("1998-12-31").cast("date")

fb_flagged = fb_flagged.withColumn(
    "is_invalid_date_departure",
    when(
        col("fb_departuredate").isNull() | (col("fb_departuredate") < min_date) | (col("fb_departuredate") > max_date),
        lit(True),
    ).otherwise(lit(False)),
)

fb_flagged = fb_flagged.withColumn(
    "is_invalid_date_commit",
    when(
        col("fb_commitdate").isNull() | (col("fb_commitdate") < min_date) | (col("fb_commitdate") > max_date),
        lit(True),
    ).otherwise(lit(False)),
)

fb_flagged = fb_flagged.withColumn(
    "is_invalid_date_arrival",
    when(
        col("fb_arrivaldate").isNull() | (col("fb_arrivaldate") < min_date) | (col("fb_arrivaldate") > max_date),
        lit(True),
    ).otherwise(lit(False)),
)

# Cross-column: arrival must be on or after departure (same row, no join needed)
fb_flagged = fb_flagged.withColumn(
    "is_arrival_before_departure",
    when(
        col("fb_departuredate").isNotNull() & col("fb_arrivaldate").isNotNull()
        & (col("fb_arrivaldate") < col("fb_departuredate")),
        lit(True),
    ).otherwise(lit(False)),
)

invalid_date_departure_count = fb_flagged.filter(col("is_invalid_date_departure")).count()
invalid_date_commit_count = fb_flagged.filter(col("is_invalid_date_commit")).count()
invalid_date_arrival_count = fb_flagged.filter(col("is_invalid_date_arrival")).count()
arrival_before_departure_count = fb_flagged.filter(col("is_arrival_before_departure")).count()

print(f"INVALID_DATE_DEPARTURE failures: {invalid_date_departure_count}")
print(f"INVALID_DATE_COMMIT failures:    {invalid_date_commit_count}")
print(f"INVALID_DATE_ARRIVAL failures:   {invalid_date_arrival_count}")
print(f"ARRIVAL_BEFORE_DEPARTURE failures: {arrival_before_departure_count}")

In [0]:
# ── Step 5: Referential Integrity Validation (§3.4) ──
# FK rules (parent-before-child order — all parents already validated in Silver):
#   fb_bookingid  -> silver.booking.bk_bookingid
#   fb_routeid    -> silver.route.ro_routeid
#   fb_airlineid  -> silver.airline.al_airlineid
# Cross-column date rule (§3.3): fb_departuredate >= bk_bookingdate from parent booking.
# Failure codes: INVALID_FK_BOOKING, INVALID_FK_ROUTE, INVALID_FK_AIRLINE, DEPARTURE_BEFORE_BOOKING

# Collect valid parent IDs for broadcast join-friendly lookups
valid_booking_ids = (
    spark.table("skyledger_airlines.silver.booking")
    .select(col("bk_bookingid").cast("int").alias("bk_bookingid"), col("bk_bookingdate"))
)
valid_route_ids = (
    spark.table("skyledger_airlines.silver.route")
    .select(col("ro_routeid").cast("int").alias("ro_routeid"))
)
valid_airline_ids = (
    spark.table("skyledger_airlines.silver.airline")
    .select(col("al_airlineid").cast("int").alias("al_airlineid"))
)

# Left join with silver.booking to get bk_bookingdate for cross-column check
fb_flagged = (
    fb_flagged
    .join(valid_booking_ids, fb_flagged.fb_bookingid == valid_booking_ids.bk_bookingid, "left")
    .join(valid_route_ids, fb_flagged.fb_routeid == valid_route_ids.ro_routeid, "left")
    .join(valid_airline_ids, fb_flagged.fb_airlineid == valid_airline_ids.al_airlineid, "left")
)

# FK validation flags
fb_flagged = fb_flagged.withColumn(
    "is_invalid_fk_booking",
    when(col("bk_bookingid").isNull(), lit(True)).otherwise(lit(False)),
)

fb_flagged = fb_flagged.withColumn(
    "is_invalid_fk_route",
    when(col("ro_routeid").isNull(), lit(True)).otherwise(lit(False)),
)

fb_flagged = fb_flagged.withColumn(
    "is_invalid_fk_airline",
    when(col("al_airlineid").isNull(), lit(True)).otherwise(lit(False)),
)

# Cross-column date check: departure must be on or after booking date
# Only applies when the booking FK is valid (bk_bookingdate is not null)
fb_flagged = fb_flagged.withColumn(
    "is_departure_before_booking",
    when(
        col("bk_bookingdate").isNotNull()
        & col("fb_departuredate").isNotNull()
        & (col("fb_departuredate") < col("bk_bookingdate")),
        lit(True),
    ).otherwise(lit(False)),
)

invalid_fk_booking_count = fb_flagged.filter(col("is_invalid_fk_booking")).count()
invalid_fk_route_count = fb_flagged.filter(col("is_invalid_fk_route")).count()
invalid_fk_airline_count = fb_flagged.filter(col("is_invalid_fk_airline")).count()
departure_before_booking_count = fb_flagged.filter(col("is_departure_before_booking")).count()

print(f"INVALID_FK_BOOKING failures:       {invalid_fk_booking_count}")
print(f"INVALID_FK_ROUTE failures:         {invalid_fk_route_count}")
print(f"INVALID_FK_AIRLINE failures:       {invalid_fk_airline_count}")
print(f"DEPARTURE_BEFORE_BOOKING failures: {departure_before_booking_count}")

In [0]:
# ── Step 6: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──

# Build comma-separated rejection_reason from all flag columns
fb_flagged = fb_flagged.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("is_null_pk"), lit("NULL_PK")),
        when(col("is_duplicate_pk"), lit("DUPLICATE_PK")),
        when(col("is_invalid_ticketprice"), lit("INVALID_TICKETPRICE")),
        when(col("is_invalid_discount"), lit("INVALID_DISCOUNT")),
        when(col("is_invalid_seats"), lit("INVALID_SEATS")),
        when(col("is_invalid_date_departure"), lit("INVALID_DATE_DEPARTURE")),
        when(col("is_invalid_date_commit"), lit("INVALID_DATE_COMMIT")),
        when(col("is_invalid_date_arrival"), lit("INVALID_DATE_ARRIVAL")),
        when(col("is_arrival_before_departure"), lit("ARRIVAL_BEFORE_DEPARTURE")),
        when(col("is_invalid_fk_booking"), lit("INVALID_FK_BOOKING")),
        when(col("is_invalid_fk_route"), lit("INVALID_FK_ROUTE")),
        when(col("is_invalid_fk_airline"), lit("INVALID_FK_AIRLINE")),
        when(col("is_departure_before_booking"), lit("DEPARTURE_BEFORE_BOOKING")),
    ),
)

# Silver columns (business data only — no flags, no join helper columns)
silver_cols = [
    "fb_bookingid", "fb_routeid", "fb_airlineid", "fb_linenumber",
    "fb_seats", "fb_ticketprice", "fb_discount", "fb_tax",
    "fb_refundflag", "fb_segmentstatus",
    "fb_departuredate", "fb_commitdate", "fb_arrivaldate",
    "fb_cabinclass", "fb_comment",
]

# Clean records: passed every check (rejection_reason is empty string)
silver_fb = fb_flagged.filter(col("rejection_reason") == "").select(*silver_cols)

# Quarantined records: failed one or more checks — carry rejection_reason
quarantine_cols = silver_cols + ["rejection_reason"]
quarantine_fb = fb_flagged.filter(col("rejection_reason") != "").select(*quarantine_cols)

silver_count = silver_fb.count()
quarantine_count = quarantine_fb.count()

print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# Write Silver table (overwrite for idempotency — same result on every run)
(silver_fb.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.flight_booking"))

# Write Quarantine table (overwrite for idempotency)
(quarantine_fb.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.flight_booking_quarantine"))

print("Tables written: silver.flight_booking, silver.flight_booking_quarantine")

In [0]:
# ── Step 7: Bronze-Silver-Quarantine Reconciliation (§3.5) ──
# Bronze row count must equal Silver + Quarantine for the flight_booking table.

bronze_total = spark.table("skyledger_airlines.bronze.flight_booking").count()
silver_total = spark.table("skyledger_airlines.silver.flight_booking").count()
quarantine_total = spark.table("skyledger_airlines.silver.flight_booking_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != "
    f"Silver ({silver_total}) + Quarantine ({quarantine_total})"
)

print("Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 8: Data Quality Report ──
# Documents every check executed, even when zero rows fail.

dq_report = spark.createDataFrame(
    [
        ("flight_booking", "NULL_PK", "fb_bookingid and fb_linenumber must both be not null", null_pk_count, "Quarantine"),
        ("flight_booking", "DUPLICATE_PK", "composite key (fb_bookingid, fb_linenumber) must be unique", duplicate_pk_count, "Quarantine"),
        ("flight_booking", "INVALID_TICKETPRICE", "fb_ticketprice must be strictly greater than zero", invalid_ticketprice_count, "Quarantine"),
        ("flight_booking", "INVALID_DISCOUNT", "fb_discount must lie in [0, 1]", invalid_discount_count, "Quarantine"),
        ("flight_booking", "INVALID_SEATS", "fb_seats must be a positive integer (>= 1)", invalid_seats_count, "Quarantine"),
        ("flight_booking", "INVALID_DATE_DEPARTURE", "fb_departuredate must fall within 1994-01-01 to 1998-12-31", invalid_date_departure_count, "Quarantine"),
        ("flight_booking", "INVALID_DATE_COMMIT", "fb_commitdate must fall within 1994-01-01 to 1998-12-31", invalid_date_commit_count, "Quarantine"),
        ("flight_booking", "INVALID_DATE_ARRIVAL", "fb_arrivaldate must fall within 1994-01-01 to 1998-12-31", invalid_date_arrival_count, "Quarantine"),
        ("flight_booking", "ARRIVAL_BEFORE_DEPARTURE", "fb_arrivaldate must be on or after fb_departuredate", arrival_before_departure_count, "Quarantine"),
        ("flight_booking", "DEPARTURE_BEFORE_BOOKING", "fb_departuredate must be on or after parent bk_bookingdate", departure_before_booking_count, "Quarantine"),
        ("flight_booking", "INVALID_FK_BOOKING", "fb_bookingid must resolve to a valid booking in silver.booking", invalid_fk_booking_count, "Quarantine"),
        ("flight_booking", "INVALID_FK_ROUTE", "fb_routeid must resolve to a valid route in silver.route", invalid_fk_route_count, "Quarantine"),
        ("flight_booking", "INVALID_FK_AIRLINE", "fb_airlineid must resolve to a valid airline in silver.airline", invalid_fk_airline_count, "Quarantine"),
    ],
    ["table", "rule_code", "check_description", "failing_rows", "action_taken"],
)

display(dq_report)

# Show quarantined records if any exist
if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table("skyledger_airlines.silver.flight_booking_quarantine"))
else:
    print("\nNo quarantined records — all flight_booking rows passed every check.")

# Show final Silver table
print(f"\n── Silver Flight Booking Table ({silver_count} rows) ──")
display(spark.table("skyledger_airlines.silver.flight_booking"))